# ViT Assignment



## PART 1 Setup Directory

Run this cell first to mount Google Drive and set the working directory to your **CIS515** folder.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Set working directory to CIS515 folder on Google Drive
COLAB_DIR = '/content/drive/MyDrive/CIS515'
os.makedirs(COLAB_DIR, exist_ok=True)
os.chdir(COLAB_DIR)

print(f"Working directory: {os.getcwd()}")
print("Files in CIS515 folder:")
print(os.listdir(COLAB_DIR))

Mounted at /content/drive
Working directory: /content/drive/MyDrive/CIS515
Files in CIS515 folder:
['competition_data']


In [4]:
# Install dependencies
!pip install transformers datasets optuna torch torchvision accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 28.8 MB/s eta 0:00:00


In [3]:
from datasets import load_dataset, concatenate_datasets

print("Loading raw images from HuggingFace...")
raw = load_dataset("dpdl-benchmark/oxford_flowers102")

all_data = concatenate_datasets([raw["train"], raw["validation"], raw["test"]])

Loading raw images from HuggingFace...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/418M [00:00<?, ?B/s]

data/test-00000-of-00006.parquet:   0%|          | 0.00/420M [00:00<?, ?B/s]

data/test-00001-of-00006.parquet:   0%|          | 0.00/416M [00:00<?, ?B/s]

...

data/test-00005-of-00006.parquet:   0%|          | 0.00/418M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/416M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1020 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6149 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1020 [00:00<?, ? examples/s]

## Import Libraries and setup GPU

In [5]:
import torch
import optuna
import numpy as np
from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import (
    ViTForImageClassification,
    ViTImageProcessor,
    get_scheduler,
)


N_CLASSES  = 102
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

Using device: cuda


##Load & Preprocess Dataset

In [6]:
import pandas as pd
import torch
from torch.utils.data import TensorDataset
from datasets import load_dataset, concatenate_datasets

################ PART 1 ###############
#(1) Enter Your Code Here
#(1.2) load ViT Image processor
# Load pretrained ViT
MODEL_NAME = "google/vit-base-patch16-224"
processor  = ViTImageProcessor.from_pretrained(MODEL_NAME)
########################################

COMPETITION_DIR = "/content/drive/MyDrive/CIS515/competition_data/public"

from torch.utils.data import Dataset

class FlowerDataset(Dataset):
    def __init__(self, csv_path, all_data, processor, has_labels=True):
        self.df         = pd.read_csv(csv_path)
        self.indices    = self.df["image_id"].str.replace("img_", "").astype(int).tolist()
        self.all_data   = all_data
        self.processor  = processor
        self.has_labels = has_labels

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        image = self.all_data[self.indices[idx]]["image"].convert("RGB")
        enc   = self.processor(images=image, return_tensors="pt")
        pv    = enc["pixel_values"].squeeze(0)   # (3, 224, 224)

        if self.has_labels:
            label = int(self.df["label"].iloc[idx])
            return pv, torch.tensor(label)
        return (pv,)


train_ds = FlowerDataset(f"{COMPETITION_DIR}/train.csv", all_data, processor)
val_ds   = FlowerDataset(f"{COMPETITION_DIR}/val.csv",   all_data, processor)
test_ds  = FlowerDataset(f"{COMPETITION_DIR}/test_public.csv", all_data, processor, has_labels=False)

print(f"\nTrain: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]


Train: 4913 | Val: 1228 | Test: 1228


In [6]:
import pandas as pd
import torch
from torch.utils.data import Dataset
from torchvision import transforms
from transformers import ViTImageProcessor

################ PART 1.2 ###############
# Load the ViT processor and pull its normalization constants. We use
# these in torchvision pipelines so we can add augmentation for training.
MODEL_NAME = "google/vit-base-patch16-224"
processor  = ViTImageProcessor.from_pretrained(MODEL_NAME)
IMG_MEAN   = processor.image_mean
IMG_STD    = processor.image_std
IMG_SIZE   = processor.size["height"]   # 224
########################################

COMPETITION_DIR = "/content/drive/MyDrive/CIS515/competition_data/public"


# Training pipeline: random crop/flip/color-jitter/RandAugment.
# The goal is to fight the overfitting we see in the current run --
# train loss was crashing to 0.058 while val accuracy stuck at 99.3%.
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.RandAugment(num_ops=2, magnitude=9),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMG_MEAN, std=IMG_STD),
])

# Val / test pipeline: deterministic resize + center crop, no randomness.
eval_transforms = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 256 / 224)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMG_MEAN, std=IMG_STD),
])


class FlowerDataset(Dataset):
    def __init__(self, csv_path, all_data, transform, has_labels=True):
        self.df         = pd.read_csv(csv_path)
        self.indices    = self.df["image_id"].str.replace("img_", "").astype(int).tolist()
        self.all_data   = all_data
        self.transform  = transform
        self.has_labels = has_labels

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        image = self.all_data[self.indices[idx]]["image"].convert("RGB")
        pixel_values = self.transform(image)

        if self.has_labels:
            label = int(self.df["label"].iloc[idx])
            return pixel_values, torch.tensor(label)
        return (pixel_values,)


train_ds = FlowerDataset(f"{COMPETITION_DIR}/train.csv",       all_data, train_transforms)
val_ds   = FlowerDataset(f"{COMPETITION_DIR}/val.csv",         all_data, eval_transforms)
test_ds  = FlowerDataset(f"{COMPETITION_DIR}/test_public.csv", all_data, eval_transforms, has_labels=False)

print(f"\nTrain: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")


preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]


Train: 4913 | Val: 1228 | Test: 1228


## Training Function

In [22]:
import copy
    # Starter values
BATCH_SIZE   = 32
N_UNFREEZE   = 10  # allows to unfreeze last N_UNFREEZE encoder blocks
WEIGHT_DECAY = 0.01
LR=1e-2
WARMUP_RATIO=0.0
SCHEDULER_TYPE="constant_with_warmup"
EPOCHS     = 12
N_TRIALS   = 15

def train_and_evaluate(


    lr: float = LR,
    warmup_ratio: float = WARMUP_RATIO,
    scheduler_type: str = SCHEDULER_TYPE,

    n_unfreeze: int     = N_UNFREEZE,
    weight_decay: float = WEIGHT_DECAY,
    batch_size: int     = BATCH_SIZE,


    trial               = None,


):


    #train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    #val_loader   = DataLoader(val_ds,   batch_size=64)

    # MAKE THE LOADERS MORE EFFICIENT

    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True,
        num_workers=4, pin_memory=True, persistent_workers=True,
    )
    val_loader = DataLoader(
        val_ds, batch_size=64,
        num_workers=2, pin_memory=True, persistent_workers=True,
    )





    model = ViTForImageClassification.from_pretrained(
        MODEL_NAME,
        num_labels=N_CLASSES,
        ignore_mismatched_sizes=True,
    ).to(DEVICE)


    for param in model.vit.parameters():
        param.requires_grad = False


    for param in model.classifier.parameters():
        param.requires_grad = True



    if n_unfreeze > 0:
        for block in model.vit.encoder.layer[-n_unfreeze:]:
            for param in block.parameters():
                param.requires_grad = True

    for param in model.vit.layernorm.parameters():
        param.requires_grad = True

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"  Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

    optimizer = AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=lr,
        weight_decay=weight_decay,
    )

    total_steps  = len(train_loader) * EPOCHS
    warmup_steps = int(total_steps * warmup_ratio)

    scheduler = get_scheduler(
        scheduler_type,
        optimizer=optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    best_val_acc    = 0.0
    best_model_state = None

    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0
        for batch in train_loader:
            pixel_values, labels = batch[0].to(DEVICE), batch[1].to(DEVICE)
            outputs = model(pixel_values=pixel_values, labels=labels)
            loss         = outputs.loss
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()

        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for batch in val_loader:
                pixel_values, labels = batch[0].to(DEVICE), batch[1].to(DEVICE)
                preds   = model(pixel_values=pixel_values).logits.argmax(dim=-1)
                correct += (preds == labels).sum().item()
                total   += labels.size(0)


        val_acc = correct / total
        if val_acc >= best_val_acc:
            best_val_acc     = val_acc
            best_model_state = copy.deepcopy(model.state_dict())

        print(f"    Epoch {epoch+1}/{EPOCHS} | loss={total_loss/len(train_loader):.3f} | val_acc={val_acc:.3f}")


        if trial is not None:
            # this was a dup - found as adding inefficiency
            #trial.report(val_acc, epoch)

            trial.report(val_acc, epoch)
            if trial.should_prune():
                print("Trial pruned.")
                raise optuna.exceptions.TrialPruned()

    model.load_state_dict(best_model_state)

    return  best_val_acc, model

## PART 2: Baseline Model Performance

In [8]:
print("========================================")
print("BASELINE hyperparameters")
print("========================================")


base_acc, _ = train_and_evaluate(
    lr=LR,
    warmup_ratio=WARMUP_RATIO,
    scheduler_type=SCHEDULER_TYPE,
)
print(f"\nBASELINE config val accuracy: {base_acc*100:.1f}%")

BASELINE hyperparameters


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                          
------------------+----------+------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000, 768]) vs model:torch.Size([102, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000]) vs model:torch.Size([102])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


  Trainable: 70,958,694 / 85,877,094 (82.6%)
    Epoch 1/15 | loss=4.437 | val_acc=0.107
    Epoch 2/15 | loss=3.564 | val_acc=0.248
    Epoch 3/15 | loss=2.905 | val_acc=0.360
    Epoch 4/15 | loss=2.522 | val_acc=0.410
    Epoch 5/15 | loss=2.486 | val_acc=0.448
    Epoch 6/15 | loss=2.266 | val_acc=0.495
    Epoch 7/15 | loss=2.333 | val_acc=0.468
    Epoch 8/15 | loss=2.277 | val_acc=0.507
    Epoch 9/15 | loss=2.209 | val_acc=0.429
    Epoch 10/15 | loss=2.411 | val_acc=0.448
    Epoch 11/15 | loss=2.565 | val_acc=0.376
    Epoch 12/15 | loss=2.729 | val_acc=0.302
    Epoch 13/15 | loss=2.700 | val_acc=0.429
    Epoch 14/15 | loss=2.573 | val_acc=0.409
    Epoch 15/15 | loss=2.381 | val_acc=0.369

BASELINE config val accuracy: 50.7%


#PART 3 Run Optuna Study

In [10]:
def objective(trial: optuna.Trial) -> float:

    # Existing hyperparameters (with updated ranges)
    lr             = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
    warmup_ratio   = trial.suggest_float("warmup_ratio", 0.0, 0.3)
    scheduler_type = trial.suggest_categorical("scheduler_type",
                         ["linear", "cosine", "constant_with_warmup"])

    #(3.1) ENTER YOUR CODE HERE: EXPLORE ADDITIONAL HYPERPARAMETERS AND HYPERPARAMTER VALUES###
    n_unfreeze     = trial.suggest_int("n_unfreeze", 1, 12)
    weight_decay   = trial.suggest_float("weight_decay", 1e-4, 0.1, log=True)
    batch_size     = trial.suggest_categorical("batch_size", [16, 32, 64])

    print(f"\nTrial {trial.number}: lr={lr:.2e}, warmup={warmup_ratio:.2f}, "
          f"sched={scheduler_type}, unfreeze={n_unfreeze}, "
          f"wd={weight_decay:.4f}, bs={batch_size}") #####PRINT HYPERPARAMETERS HERE##########

    val_acc, _ = train_and_evaluate(
        lr=lr, warmup_ratio=warmup_ratio, scheduler_type=scheduler_type,
        n_unfreeze=n_unfreeze, weight_decay=weight_decay, batch_size=batch_size,
        trial=trial, ######## (3.1) Update this function to use all hyperparameteres in trials
    )
    return val_acc

'''
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=1),
)
study.optimize(objective, n_trials=N_TRIALS)
'''

# was having hanging issues w/hf - this makes the study resumable
# also tune the warm up steps for higher epoch count
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=3),
    storage="sqlite:////content/drive/MyDrive/CIS515/optuna_study.db",
    study_name="vit_flowers_v1",
    load_if_exists=True,
)
study.optimize(objective, n_trials=N_TRIALS)

[I 2026-04-10 19:18:48,155] A new study created in RDB with name: vit_flowers_v1



Trial 0: lr=5.61e-05, warmup=0.29, sched=linear, unfreeze=2, wd=0.0001, bs=16


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                          
------------------+----------+------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000, 768]) vs model:torch.Size([102, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000]) vs model:torch.Size([102])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


  Trainable: 14,255,718 / 85,877,094 (16.6%)
    Epoch 1/12 | loss=4.453 | val_acc=0.327
    Epoch 2/12 | loss=2.825 | val_acc=0.919
    Epoch 3/12 | loss=0.868 | val_acc=0.989
    Epoch 4/12 | loss=0.227 | val_acc=0.997
    Epoch 5/12 | loss=0.135 | val_acc=0.998
    Epoch 6/12 | loss=0.103 | val_acc=0.997
    Epoch 7/12 | loss=0.077 | val_acc=0.996
    Epoch 8/12 | loss=0.069 | val_acc=0.996
    Epoch 9/12 | loss=0.060 | val_acc=0.997
    Epoch 10/12 | loss=0.057 | val_acc=0.998
    Epoch 11/12 | loss=0.051 | val_acc=0.998
    Epoch 12/12 | loss=0.056 | val_acc=0.998


...

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                          
------------------+----------+------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000, 768]) vs model:torch.Size([102, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000]) vs model:torch.Size([102])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


  Trainable: 21,343,590 / 85,877,094 (24.9%)
    Epoch 1/12 | loss=4.194 | val_acc=0.708
    Epoch 2/12 | loss=1.454 | val_acc=0.992
    Epoch 3/12 | loss=0.191 | val_acc=0.996
    Epoch 4/12 | loss=0.098 | val_acc=0.995
    Epoch 5/12 | loss=0.061 | val_acc=0.995
    Epoch 6/12 | loss=0.040 | val_acc=0.996
    Epoch 7/12 | loss=0.036 | val_acc=0.995
    Epoch 8/12 | loss=0.032 | val_acc=0.995
    Epoch 9/12 | loss=0.026 | val_acc=0.995
    Epoch 10/12 | loss=0.026 | val_acc=0.995
    Epoch 11/12 | loss=0.029 | val_acc=0.995


[I 2026-04-10 20:55:06,105] Trial 14 finished with value: 0.995928338762215 and parameters: {'lr': 8.54376462842966e-05, 'warmup_ratio': 0.2403403685167038, 'scheduler_type': 'cosine', 'n_unfreeze': 3, 'weight_decay': 0.08887892894296451, 'batch_size': 16}. Best is trial 0 with value: 0.997557003257329.


    Epoch 12/12 | loss=0.025 | val_acc=0.995


##Best Model vs Baseline Model Comparison

In [11]:
print("=" * 55)
print("RESULTS SUMMARY")
print("=" * 55)
print(f"\nBASE  hyperparams → val accuracy: {base_acc*100:.1f}%")
print(f"BEST hyperparams → val accuracy: {study.best_value*100:.1f}%")
print(f"\nBest params found:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

RESULTS SUMMARY

BASE  hyperparams → val accuracy: 50.7%
BEST hyperparams → val accuracy: 99.8%

Best params found:
  lr: 5.6115164153345e-05
  warmup_ratio: 0.2852142919229748
  scheduler_type: linear
  n_unfreeze: 2
  weight_decay: 0.00014936568554617635
  batch_size: 16


# PART 4 Final Evaluation

In [12]:
best = study.best_params

#####################PART4#############
best_val_acc, best_model = train_and_evaluate(
    lr=best["lr"],
    warmup_ratio=best["warmup_ratio"],
    scheduler_type=best["scheduler_type"],

   ######ADD YOUR FINAL HYPERPARAMETERS HERE
    n_unfreeze    = best["n_unfreeze"],
    weight_decay  = best["weight_decay"],
    batch_size    = best["batch_size"],
)

print(f"\nFinal TEST accuracy: {best_val_acc*100:.1f}%")


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                          
------------------+----------+------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000, 768]) vs model:torch.Size([102, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000]) vs model:torch.Size([102])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


  Trainable: 14,255,718 / 85,877,094 (16.6%)
    Epoch 1/12 | loss=4.483 | val_acc=0.297
    Epoch 2/12 | loss=2.907 | val_acc=0.906
    Epoch 3/12 | loss=0.847 | val_acc=0.991
    Epoch 4/12 | loss=0.246 | val_acc=0.995
    Epoch 5/12 | loss=0.141 | val_acc=0.995
    Epoch 6/12 | loss=0.098 | val_acc=0.996
    Epoch 7/12 | loss=0.085 | val_acc=0.995
    Epoch 8/12 | loss=0.065 | val_acc=0.995
    Epoch 9/12 | loss=0.065 | val_acc=0.995
    Epoch 10/12 | loss=0.060 | val_acc=0.996
    Epoch 11/12 | loss=0.059 | val_acc=0.995
    Epoch 12/12 | loss=0.057 | val_acc=0.995

Final TEST accuracy: 99.6%


In [15]:
# Step 1: train 3 models with the same winning config
models = []
for i in range(3):
    _, model = train_and_evaluate(**study.best_params)
    models.append(model)
# Now `models` is a list of 3 trained ViT models


# Step 2: prediction function that combines all 3
def predict_ensemble(models, pixel_values):
    # `pixel_values` is a batch of images, shape (batch, 3, 224, 224)
    probs = 0
    for m in models:
        m.eval()
        with torch.no_grad():
            # Each model outputs a (batch, 102) probability vector
            probs = probs + m(pixel_values=pixel_values).logits.softmax(dim=-1)
    # Divide by 3 to get the average
    return probs / len(models)


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                          
------------------+----------+------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000, 768]) vs model:torch.Size([102, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000]) vs model:torch.Size([102])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


  Trainable: 14,255,718 / 85,877,094 (16.6%)
    Epoch 1/12 | loss=4.583 | val_acc=0.213
    Epoch 2/12 | loss=2.994 | val_acc=0.917
    Epoch 3/12 | loss=0.889 | val_acc=0.993
    Epoch 4/12 | loss=0.229 | val_acc=0.997
    Epoch 5/12 | loss=0.132 | val_acc=0.997
    Epoch 6/12 | loss=0.099 | val_acc=0.997
    Epoch 7/12 | loss=0.085 | val_acc=0.998
    Epoch 8/12 | loss=0.063 | val_acc=0.996
    Epoch 9/12 | loss=0.067 | val_acc=0.997
    Epoch 10/12 | loss=0.060 | val_acc=0.997
    Epoch 11/12 | loss=0.056 | val_acc=0.997
    Epoch 12/12 | loss=0.048 | val_acc=0.997


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                          
------------------+----------+------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000, 768]) vs model:torch.Size([102, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000]) vs model:torch.Size([102])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


  Trainable: 14,255,718 / 85,877,094 (16.6%)
    Epoch 1/12 | loss=4.478 | val_acc=0.305
    Epoch 2/12 | loss=2.876 | val_acc=0.919
    Epoch 3/12 | loss=0.913 | val_acc=0.993
    Epoch 4/12 | loss=0.244 | val_acc=0.995
    Epoch 5/12 | loss=0.138 | val_acc=0.996
    Epoch 6/12 | loss=0.092 | val_acc=0.998
    Epoch 7/12 | loss=0.079 | val_acc=0.997
    Epoch 8/12 | loss=0.075 | val_acc=0.996
    Epoch 9/12 | loss=0.067 | val_acc=0.998
    Epoch 10/12 | loss=0.057 | val_acc=0.996
    Epoch 11/12 | loss=0.054 | val_acc=0.996
    Epoch 12/12 | loss=0.052 | val_acc=0.996


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                          
------------------+----------+------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000, 768]) vs model:torch.Size([102, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000]) vs model:torch.Size([102])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


  Trainable: 14,255,718 / 85,877,094 (16.6%)
    Epoch 1/12 | loss=4.517 | val_acc=0.300
    Epoch 2/12 | loss=2.895 | val_acc=0.907
    Epoch 3/12 | loss=0.870 | val_acc=0.992
    Epoch 4/12 | loss=0.226 | val_acc=0.996
    Epoch 5/12 | loss=0.142 | val_acc=0.996
    Epoch 6/12 | loss=0.105 | val_acc=0.998
    Epoch 7/12 | loss=0.074 | val_acc=0.996
    Epoch 8/12 | loss=0.084 | val_acc=0.995
    Epoch 9/12 | loss=0.063 | val_acc=0.997
    Epoch 10/12 | loss=0.059 | val_acc=0.996
    Epoch 11/12 | loss=0.063 | val_acc=0.997
    Epoch 12/12 | loss=0.057 | val_acc=0.997


In [16]:
# Full precision
print(f"Final val accuracy: {best_val_acc*100:.4f}%")
print(f"Raw value: {best_val_acc}")

# And while you're at it, Optuna's best trial result:
print(f"\nOptuna best: {study.best_value*100:.4f}%")
print(f"Optuna best raw: {study.best_value}")


Final val accuracy: 99.5928%
Raw value: 0.995928338762215

Optuna best: 99.7557%
Optuna best raw: 0.997557003257329


In [17]:
val_size = len(val_ds)
n_correct = round(best_val_acc * val_size)
n_wrong   = val_size - n_correct
print(f"Val set size: {val_size}")
print(f"Correct: {n_correct} / {val_size}")
print(f"Wrong:   {n_wrong}")


Val set size: 1228
Correct: 1223 / 1228
Wrong:   5


In [21]:
import torch.nn.functional as F

def evaluate_with_tta(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in loader:
            pixel_values, labels = batch[0].to(device), batch[1].to(device)
            # Original + horizontal flip, averaged in probability space
            probs_orig = F.softmax(model(pixel_values=pixel_values).logits, dim=-1)
            probs_flip = F.softmax(model(pixel_values=torch.flip(pixel_values, dims=[-1])).logits, dim=-1)
            probs = (probs_orig + probs_flip) / 2
            correct += (probs.argmax(dim=-1) == labels).sum().item()
            total   += labels.size(0)
    return correct / total

val_loader_eval = DataLoader(val_ds, batch_size=64, num_workers=2, pin_memory=True)
tta_val_acc = evaluate_with_tta(best_model, val_loader_eval, DEVICE)
print(f"Val accuracy with TTA: {tta_val_acc*100:.4f}%")
print(f"Raw: {tta_val_acc}")


Val accuracy with TTA: 99.5928%
Raw: 0.995928338762215


##PART 5 Prediction on the Test Set
Saves your predictions CSV to `MyDrive/CIS515/competition_data/submissions/`.

In [13]:

##################REPLACE WITH YOUR ASU ID###########
STUDENT_NAME       = "jdolliso"

In [26]:
import pandas as pd
import os
######################PART 4 #########################

test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)
# One loader, on test_ds, no shuffle
test_loader = DataLoader(
    test_ds, batch_size=64, shuffle=False,
    num_workers=2, pin_memory=True,
)


best_model.eval()
all_preds = []
with torch.no_grad():
    for batch in test_loader:
        pixel_values = batch[0].to(DEVICE)  # TensorDataset returns tuples, not dicts
        #preds = best_model(pixel_values=pixel_values).logits.argmax(dim=-1)
        #all_preds.extend(preds.cpu().tolist())
        #probs = predict_ensemble(models, pixel_values)   # ← ensemble instead of single model
        #all_preds.extend(probs.argmax(dim=-1).cpu().tolist())

        # TTA: original + horizontal flip, averaged in probability space
        # NOTE: use best_model, not `model`
        probs_orig = F.softmax(best_model(pixel_values=pixel_values).logits, dim=-1)
        probs_flip = F.softmax(
            best_model(pixel_values=torch.flip(pixel_values, dims=[-1])).logits,
            dim=-1,
        )
        probs = (probs_orig + probs_flip) / 2
        all_preds.extend(probs.argmax(dim=-1).cpu().tolist())


test_csv  = pd.read_csv(f"{COMPETITION_DIR}/test_public.csv")
image_ids = test_csv["image_id"].tolist()
submission = pd.DataFrame({"image_id": image_ids, "label": all_preds})

submission_path = f"/content/drive/MyDrive/CIS515/competition_data/submissions/{STUDENT_NAME}_public_.csv"

os.makedirs(os.path.dirname(submission_path), exist_ok=True)
submission.to_csv(submission_path, index=False)
print(f" Saved {len(submission)} predictions → {submission_path}")
print(submission.head())

 Saved 1228 predictions → /content/drive/MyDrive/CIS515/competition_data/submissions/jdolliso_public_.csv
    image_id  label
0  img_06990      4
1  img_03068     76
2  img_01588     38
3  img_02194     24
4  img_06483      7


**Kaggle Competition Submission**

In [27]:
import os
from google.colab import userdata
#######################PART 5#################

COMPETITION_ID = "cis-515-vi-t-assignment"  # do not change this, it is the competition id
SUBMISSION_MSG = f"ViT fine-tune — {STUDENT_NAME}"

os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_API_TOKEN"]      = userdata.get("KAGGLE_API_TOKEN")

!kaggle competitions submit \
    -c {COMPETITION_ID} \
    -f "{submission_path}" \
    -m "{SUBMISSION_MSG}"

print(f"\nSubmitted: Check your score at:")
print(f"   https://www.kaggle.com/competitions/{COMPETITION_ID}/leaderboard")

100% 15.5k/15.5k [00:01<00:00, 9.46kB/s]
Successfully submitted to CIS 515 ViT Assignment
Submitted: Check your score at:
   https://www.kaggle.com/competitions/cis-515-vi-t-assignment/leaderboard


In [20]:
# Run this to see what the student CSV looks like
print(submission.head())
print(submission.columns.tolist())

    image_id  label
0  img_06990      4
1  img_03068     76
2  img_01588     38
3  img_02194     24
4  img_06483      7
['image_id', 'label']
